# A/B Test Analysis
We're going to conduct an Independent Samples T-test to analyse our A/B test. An Indepdent Samples T-test compares the differences between two means of two different samples

In [4]:
import pandas as pd
import numpy as np
import scipy.stats as stats

Export your results to a .csv file and save it to you github repository. Import your .csv file, inspect it, and clean it where neccesary.

In [5]:
# If on Github, load your data
df_A = pd.read_csv("AB-test_HCAI_Version_A_(Sheet1).csv", encoding='latin1')
df_B = pd.read_csv("AB-test_HCAI_Version_B(Sheet1).csv", encoding='latin1')

# EDA A
df_A.info() # Is your data in the right format?
df_A.head() # Quick EDA. No? Clean it, you only want the rows and columns containing likert-score data, saved as integers.

# EDA B
df_A.info() # Is your data in the right format?
df_A.head() # Quick EDA. No? Clean it, you only want the rows and columns containing likert-score data, saved as integers.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17 entries, 0 to 16
Data columns (total 10 columns):
 #   Column                                           Non-Null Count  Dtype 
---  ------                                           --------------  ----- 
 0   Id                                               17 non-null     int64 
 1   Start time                                       17 non-null     object
 2   Completion time                                  17 non-null     object
 3   Email                                            17 non-null     object
 4   Name                                             17 non-null     object
 5   I understood what I could use the app for        17 non-null     object
 6   I found the application intuitive to use         17 non-null     object
 7   I thought the application was useful             17 non-null     object
 8   I enjoyed using the application                  17 non-null     object
 9   The app could be/is better with feedback opti

,Id,Start time,Completion time,Email,Name,I understood what I could use the app for,I found the application intuitive to use,I thought the application was useful,I enjoyed using the application,The app could be/is better with feedback option
0,1,4/9/2024 10:22,4/9/2024 10:23,222855@buas.nl,Balázs Csutár,Somewhat Disagree,Disagree,Disagree,Disagree,Disagree
1,2,4/9/2024 10:29,4/9/2024 10:33,234051@buas.nl,Nils Vos,Agree,Neutral,Agree,Neutral,Completely Disagree
2,3,4/9/2024 10:31,4/9/2024 10:34,233719@buas.nl,Anton Ovcharenko,Agree,Somewhat Agree,Agree,Somewhat Agree,Agree
3,4,4/9/2024 10:42,4/9/2024 10:43,230499@buas.nl,Erfan Salour,Somewhat Agree,Agree,Somewhat Agree,Completely Agree,Agree
4,5,4/9/2024 13:18,4/9/2024 13:19,231541@buas.nl,Jonas Vos,Neutral,Disagree,Somewhat Disagree,Disagree,Neutral


In [9]:
 # Define a mapping from Likert-scale strings to integers
likert_scale_mapping = {
    'Completely Disagree': 1,
    'Disagree': 2,
    'Somewhat Disagree': 3,
    'Neutral': 4,
    'Somewhat Agree': 5,
    'Agree': 6,
    'Completely Agree': 7
}

# Apply this mapping to the relevant columns of both dataframes
df_a_clean = df_A.replace(likert_scale_mapping)
df_b_clean = df_B.replace(likert_scale_mapping)

# Drop non-Likert scale columns
df_a_likert = df_a_clean.drop(['Id', 'Start time', 'Completion time', 'Email', 'Name'], axis=1)
df_b_likert = df_b_clean.drop(['Id', 'Start time', 'Completion time', 'Email', 'Name'], axis=1)
df_a_likert.head()
df_b_likert.head()


,I understood what I could use the app for,I found the application intuitive to use,I thought the application was useful,I enjoyed using the application,The app could be/is better with feedback option
0,7,6,4,4,4
1,6,6,6,6,5
2,6,4,5,5,4
3,6,6,6,6,4
4,7,5,7,7,7


The rest we leave for tomorrow when we actually have our data. But if you are eager to play around a bit you can simply refresh the survey and fill in a couple of responses to create an A and a B version.

Now, let's start analysing our gathered data! This block we won't dive into inferential statistics since it can get complex quite fast; we'll do that in Year 2, block A. For now, you just need to know that we need to test whether the data is normally distributed and whether the variances of both samples are equal. Otherwise, our statistical tests would not be valid and we can therefore not say that the results we're seeing are due to chance. What we are going to statistically ascertain is whether there is a statistically significant different in the mean of a given variable for version A or B. 

In [8]:
# Function to run Shapiro-Wilk test on each column of a given DataFrame
def run_shapiro_wilk_test(df):
    for column in df.columns:
        stat, p_value = stats.shapiro(df[column])
        print(f"Shapiro-Wilk Test for '{column}': p-value = {p_value}")

# Run Shapiro-Wilk test on each question for Version A
print("Version A - Shapiro-Wilk Test Results:")
run_shapiro_wilk_test(df_a_likert)

# Add a separator for clarity
print("\n" + "-"*50 + "\n")

# Run Shapiro-Wilk test on each question for Version B
print("Version B - Shapiro-Wilk Test Results:")
run_shapiro_wilk_test(df_b_likert)


Version A - Shapiro-Wilk Test Results:
Shapiro-Wilk Test for 'I understood what I could use the app for': p-value = 0.0004901963924092932
Shapiro-Wilk Test for 'I found the application intuitive to use': p-value = 0.004948943839024762
Shapiro-Wilk Test for 'I thought the application was useful': p-value = 0.014107735074620067
Shapiro-Wilk Test for 'I enjoyed using the application': p-value = 0.02025933920239676
Shapiro-Wilk Test for 'The app could be/is better with feedback option': p-value = 0.005151216050800031

--------------------------------------------------

Version B - Shapiro-Wilk Test Results:
Shapiro-Wilk Test for 'I understood what I could use the app for': p-value = 0.007218823389877197
Shapiro-Wilk Test for 'I found the application intuitive to use': p-value = 0.002066121515663779
Shapiro-Wilk Test for 'I thought the application was useful': p-value = 0.05981320902623695
Shapiro-Wilk Test for 'I enjoyed using the application': p-value = 0.010562372017855556
Shapiro-Wilk T

In [12]:
# Function to check normality for all questions
def check_normality(df):
    results = {}
    for question in df.columns:
        stat, p_value = stats.shapiro(df[question])
        results[question] = 'Yes' if p_value > 0.05 else 'No'
    return results

# Function to check homogeneity of variances for all questions
def check_homogeneity(df1, df2):
    results = {}
    for question in df1.columns:
        stat, p_value = stats.levene(df1[question], df2[question])
        results[question] = 'Yes' if p_value > 0.05 else 'No'
    return results

# Running the normality check for both datasets
normality_a = check_normality(df_a_likert)
normality_b = check_normality(df_b_likert)

# Running the homogeneity check
homogeneity_results = check_homogeneity(df_a_likert, df_b_likert)

# Output the results
print("Normality Check for Version A:")
for question, result in normality_a.items():
    print(f"{question}: {result}")

print("\nNormality Check for Version B:")
for question, result in normality_b.items():
    print(f"{question}: {result}")

print("\nHomogeneity of Variances Check:")
for question, result in homogeneity_results.items():
    print(f"{question}: {result}")

Normality Check for Version A:
I understood what I could use the app for: No
I found the application intuitive to use: No
I thought the application was useful: No
I enjoyed using the application: No
The app could be/is better with feedback option: No

Normality Check for Version B:
I understood what I could use the app for: No
I found the application intuitive to use: No
I thought the application was useful: Yes
I enjoyed using the application: No
The app could be/is better with feedback option: No

Homogeneity of Variances Check:
I understood what I could use the app for: Yes
I found the application intuitive to use: Yes
I thought the application was useful: Yes
I enjoyed using the application: No
The app could be/is better with feedback option: Yes


Now that is in the right format and we know the column names. Replace 'A' with the column name which holds your original baseline version; A. Replace 'B' with the column name which holds the result of your improved version; B.

In [19]:
# List of questions
questions = ['I understood what I could use the app for', 'I found the application intuitive to use', 'I thought the application was useful', 'I enjoyed using the application', 'The app could be/is better with feedback option']  # replace with your actual question names

# Create a dictionary to store the results
results = {}
rng = np.random.default_rng()
# Iterate over each question
for question in questions:
    # Run Bootstrapped Independent Samples T-test
    t_stat, p_val = stats.ttest_ind(df_a_likert[question], df_b_likert[question], random_state=rng)
    
    # Check if the p-value is less than or equal to 0.05
    if p_val <= 0.05:
        results[question] = ('Yes', p_val)
    else:
        results[question] = ('No', p_val)

# Print the results
for question, result in results.items():
    print(f"For the question '{question}', is the difference statistically significant? {result[0]} (p-value: {result[1]})")

For the question 'I understood what I could use the app for', is the difference statistically significant? No (p-value: 0.4325211717523607)
For the question 'I found the application intuitive to use', is the difference statistically significant? No (p-value: 0.9112607450045376)
For the question 'I thought the application was useful', is the difference statistically significant? No (p-value: 0.8333833994763196)
For the question 'I enjoyed using the application', is the difference statistically significant? No (p-value: 0.39709556231248067)
For the question 'The app could be/is better with feedback option', is the difference statistically significant? No (p-value: 0.14449075332073363)


Great, that was our first t-test. Save the results to your learning log in the week 8 and interpret them there. Were they what you expected? What are you going to change to improve your design if neccesary. 